# Module 2 · Lesson 02: Few-Shot Learning

**Few-shot prompting** provides examples *within the prompt* to teach the model
a specific pattern. This dramatically improves consistency and format compliance.

## What you will learn
1. One-shot vs few-shot prompting
2. Few-shot for **classification**
3. Few-shot for **data extraction**
4. Few-shot for **text transformation**
5. Few-shot for **code style** conversion
6. Combining few-shot with system prompts
7. Best practices for example selection

In [1]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / "module_02_prompt_enginnering/.env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
OPENAI_MODEL = "gpt-4o-mini" 
if client:
    print(f"OpenAI client ready - using model {OPENAI_MODEL}")

OpenAI client ready - using model gpt-4o-mini


In [2]:
def ask(prompt: str, system_content: str= None, temperature: float= 0.7, max_resp_tokens: int= 1200, ai_model: str="gpt-4o-mini"):
    msgs = []
    if system_content:
        msgs.append({"role": "system", "content": system_content})
    msgs.append({"role":"user", "content": prompt})
    resp = client.chat.completions.create(
        model=ai_model,
        messages=msgs,
        temperature=temperature,
        max_tokens=max_resp_tokens
    )
    return resp.choices[0].message.content

---
## 1. One-Shot Example

Even **one example** establishes a pattern the model will follow:

In [3]:
prompt = """Convert informal text to formal:

Informal: "Hey, can you help me with this thing?"ArithmeticError
Formal: "Good day, would you able to assist me with this matter?"

Informal: "That's super cool, i wanna try it out asap"
Formal:
"""

result = ask(prompt, temperature=0)
display(Markdown(f"**Result:** {result}"))

**Result:** Formal: "That is quite impressive; I would like to try it at my earliest convenience."

---
## 2. Few-Shot Classification

Multiple examples make classification **more consistent**:

In [5]:
prompt_example = """Classify the support ticket priority as HIGH, MEDIUM or LOW. Response with only the priority level, nothing else.

Tickey: "My account is locked and i can't access my data."
HIGH

Ticket: "Can you update my billing address?"
LOW

Ticket: "The export feature is giving wrong numbers."
MEDIUM

Ticket: "{ticket}" """

test_tickets = [
    "Our entire system is down, productions is affected!",
    "How do i change my profile picture?",
    "CSV export is missing the last column of data.",
    "The search result seem slightly inaccurate",
]

print(f"{'Ticket':<55} {'Priority'}")
print("─" * 68)
for ticket in test_tickets:
    priority = ask(prompt_example.format(ticket=ticket), temperature=0, max_resp_tokens=5).strip()
    emoji = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}.get(priority, "⚪")
    print(f"{ticket[:52]+'...':<55} {emoji} {priority}")

Ticket                                                  Priority
────────────────────────────────────────────────────────────────────
Our entire system is down, productions is affected!...  🔴 HIGH
How do i change my profile picture?...                  🟢 LOW
CSV export is missing the last column of data....       🟡 MEDIUM
The search result seem slightly inaccurate...           🟢 LOW


---
## 3. Few-Shot Data Extraction

Examples teach the model to extract data in a **consistent format**:

In [6]:
user_data = "Premium noise-cancelling headphone by Sony, now $349 in the audio department"

prompt = """Extract product info as: NAME | PRICE | CATEGORY

Text: "The new i-phone 16 Pro is available for $999 in our electronics section."
Result: i-Phone 16 Pro | $999 | Electronics

Text: "Fresh organic avocados on sale for $2.99 per piece in the produce aisle."
Result: Organic Avocados | $2.99 | Produce

Text: {user_data}
Result:"""

result = ask(prompt.format(user_data=user_data), temperature=0, max_resp_tokens=30)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `Sony Premium Noise-Cancelling Headphones | $349 | Audio`

---
## 4. Few-Shot Text Transformation

In [7]:
prompt = """Conver technical jargon to plain English:

Technical: "The API latency is exceeding the SLA threshold."
Plain: "Our system is responding too slowly.

Technical: "We need to implment CI/CD pipeline for microservices."
Plain: "We need to automate our software deployment process."

Tecnhical: "The DNS propagation delay is causing intermitted 503 errors.
Plain:"""

result = ask(prompt, temperature=0, max_resp_tokens=30)
display(Markdown(f"**Simplified:** {result.strip()}"))

**Simplified:** "Our website is experiencing occasional downtime because it’s taking too long for changes to take effect."

In [8]:
prompt = """Convert technical jargon into funny everyday language:

Technical: "The server is not responding "ArithmeticError"
Funny: "The server has decided to ignore us like a bad text message."

Technical: "The application crashed due to an unexpected error." 
Funny: "The app saw something id didn't like and dramatically failed." 

Technical: "The password you entered is incorrect."
Funny:
"""

result = ask(prompt, max_resp_tokens=150)
display(Markdown(f"**Funny version:** {result.strip()}"))

**Funny version:** "The password you typed is playing hard to get – it’s not letting us in!"

---
## 5. Few-Shot Code Style

Teach the model to transform code following a specific style guide:

In [10]:
prompt = '''Convert to PEP-8 style with type hints and docstrings:
 
Before:
def calc(x,y): return x+y
 
After:
def calc(x: float, y: float) -> float:
    """Calculate the sum of two numbers."""
    return x + y
 
Before:
def getUser(id): return db.query("SELECT * FROM users WHERE id=" + str(id))
 
After:'''
 
result = ask(prompt, temperature=0, max_resp_tokens=200)
# display(Markdown(f"```python\n{result.strip()}\n```"))
display(Markdown(result.strip()))

Here is the `getUser` function converted to PEP-8 style with type hints and a docstring:

```python
def get_user(user_id: int) -> dict:
    """Retrieve a user from the database by their ID.

    Args:
        user_id (int): The ID of the user to retrieve.

    Returns:
        dict: A dictionary containing user information.
    """
    return db.query("SELECT * FROM users WHERE id=" + str(user_id))
```

### Changes Made:
1. **Function Name**: Changed `getUser` to `get_user` to follow the PEP-8 naming convention for functions.
2. **Parameter Name**: Changed `id` to `user_id` for clarity and to avoid shadowing the built-in `id` function.
3. **Type Hinting**: Added type hints for the parameter and return type.
4. **Docstring**: Added a docstring to describe the function's purpose, its

---
## 6. Combining Few-Shot with System Prompt

**System prompt** sets behaviour + **Few-shot** sets format = best results:

In [11]:
# ── Example 6: System + Few-shot ─────────────────────
system = """You are a Git commit message generator.
Write clear, conventional commit messages.
Format: <type>(<scope>): <description>"""
 
prompt = """Examples:
 
Change: Fixed the login button not working on mobile
Commit: fix(auth): resolve login button tap target on mobile devices
 
Change: Added dark mode support to the settings page
Commit: feat(settings): add dark mode toggle and theme persistence
 
Change: Reorganized the test files into separate folders
Commit:"""
 
result = ask(prompt, system_content=system, temperature=0)
display(Markdown(f"**Generated commit:** `{result.strip()}`"))

**Generated commit:** `chore(tests): reorganize test files into separate directories for better structure`

---
## 7. Best Practices for Example Selection

| Practice | Why |
|----------|-----|
| **Use 2–5 examples** | Enough to establish pattern, not too many to waste tokens |
| **Cover edge cases** | Include unusual inputs in your examples |
| **Be consistent** | Same format across all examples |
| **Order matters** | Put easiest examples first, hardest last |
| **Diversify** | Use varied content to avoid overfitting to one example |

---
## Exercise — Build Your Own Few-Shot Prompt 🏋️

Create a few-shot prompt for one of these tasks:
- Email subject line generator
- Bug report summarizer
- Language translation with specific tone

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **One-shot** | 1 example — establishes basic pattern |
| **Few-shot** | 2-5 examples — more consistent, better edge cases |
| **Format control** | Examples teach exact output format |
| **Classification** | Very effective with labeled examples |
| **System + Few-shot** | Combine for best results |
| **Cost** | More examples = more input tokens = higher cost |

---
**Next:** `03_chain_of_thought.ipynb` — Guide models to reason step by step